# Incidencia Delictiva en Mexico 2015 - 2025

Según cifras oficiales SESNSP

In [1]:
import pandas as pd
import polars as pl
import geopandas as gpd
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [2]:
data = pd.read_csv('municipal-2015-2025.csv', encoding='latin1')

In [3]:
delitos = pl.from_pandas(data)

In [4]:
meses = {
    "Enero": 1,
    "Febrero": 2,
    "Marzo": 3,
    "Abril": 4,
    "Mayo": 5,
    "Junio": 6,
    "Julio": 7,
    "Agosto": 8,
    "Septiembre": 9,
    "Octubre": 10,
    "Noviembre": 11,
    "Diciembre": 12,
}

In [5]:
delitos = (
    delitos
        .rename({'Cve. Municipio':'cve_geo'})
        .select(pl.exclude('Clave_Ent'))
        .unpivot(
            on=['Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre'],
            index=['Año','cve_geo','Entidad','Municipio','Bien jurídico afectado','Tipo de delito','Subtipo de delito','Modalidad'],
            variable_name='Mes',
            value_name='Incidencia',
        )
        .select(['Año','Mes','cve_geo','Entidad','Municipio','Bien jurídico afectado','Tipo de delito','Subtipo de delito','Modalidad','Incidencia'])
        .with_columns(
           pl.col('Mes').replace(meses).cast(pl.Int8).alias('Mes_num')
        )
        .with_columns(
            pl.datetime(pl.col('Año'), pl.col('Mes_num'), 1).alias('Fecha')
        )
        .select((['Fecha','cve_geo','Entidad','Municipio','Bien jurídico afectado','Tipo de delito','Subtipo de delito','Modalidad','Incidencia']))
        .filter(pl.col('Fecha') < datetime(2025, 11, 1),)
)

In [6]:
#delitos.write_csv('delitos.csv')

In [7]:
delitos.sample(10)

Fecha,cve_geo,Entidad,Municipio,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Incidencia
datetime[μs],i64,str,str,str,str,str,str,f64
2022-02-01 00:00:00,30206,"""Veracruz de Ignacio de la Llav…","""Nanchital de Lázaro Cárdenas d…","""El patrimonio""","""Otros delitos contra el patrim…","""Otros delitos contra el patrim…","""Otros delitos contra el patrim…",0.0
2017-07-01 00:00:00,8012,"""Chihuahua""","""Carichí""","""La vida y la Integridad corpor…","""Lesiones""","""Lesiones culposas""","""Con otro elemento""",0.0
2021-06-01 00:00:00,31087,"""Yucatán""","""Tetiz""","""El patrimonio""","""Robo""","""Robo a casa habitación""","""Sin violencia""",0.0
2024-03-01 00:00:00,20409,"""Oaxaca""","""Santa María del Tule""","""La vida y la Integridad corpor…","""Lesiones""","""Lesiones culposas""","""Con otro elemento""",0.0
2024-08-01 00:00:00,7021,"""Chiapas""","""Copainalá""","""La libertad y la seguridad sex…","""Incesto""","""Incesto""","""Incesto""",0.0
2018-01-01 00:00:00,20536,"""Oaxaca""","""San Vicente Nuñú""","""El patrimonio""","""Robo""","""Robo a transportista""","""Sin violencia""",0.0
2017-07-01 00:00:00,20026,"""Oaxaca""","""Chalcatongo de Hidalgo""","""El patrimonio""","""Robo""","""Robo de maquinaria""","""Robo de tractores Con violenci…",0.0
2023-09-01 00:00:00,20555,"""Oaxaca""","""Trinidad Zaachila""","""Libertad personal""","""Secuestro""","""Secuestro""","""Secuestro con calidad de rehén""",0.0
2016-02-01 00:00:00,16010,"""Michoacán de Ocampo""","""Arteaga""","""La vida y la Integridad corpor…","""Aborto""","""Aborto""","""Aborto""",0.0


In [8]:
df_geo = gpd.read_parquet('geo.parquet')

In [9]:
df_geo = df_geo[['CVEGEO','NOM_ENT','NOMGEO','AREA','PERIMETER','geometry']]

In [10]:
df_geo['NOM_ENT'] = df_geo['NOM_ENT'].replace(['Coahuila de Zaragoza','Michoacán de Ocampo','Veracruz de Ignacio de la Llave'], ['Coahuila','Michoacán','Veracruz'])

In [11]:
df_geo['geometry'].to_crs(epsg=6372)
df_geo.loc[:, 'lon'] = df_geo.geometry.centroid.x
df_geo.loc[:, 'lat'] = df_geo.geometry.centroid.y

In [12]:
gdf = df_geo.rename(columns={
    'CVEGEO':'cve_geo',
    'NOM_ENT':'entidad',
    'NOMGEO':'municipio',
    'AREA':'area',
    'PERIMETER':'perimetro',
})

In [13]:
gdf['cve_geo'] = gdf['cve_geo'].astype(int)

In [ ]:
#gdf.to_parquet('geo_mun.parquet')

In [15]:
gdf.to_file('geo_mun.geojson', driver='GeoJSON')

In [15]:
df = delitos.to_pandas()

In [16]:
df_end = (
    df.merge(
        gdf,
        on='cve_geo',
        )
)

In [17]:
df_end = df_end[['Fecha','cve_geo','Entidad','Municipio',
    'Bien jurídico afectado','Tipo de delito','Subtipo de delito',
    'Modalidad','Incidencia','geometry','lon','lat']]

In [18]:
df_end.sample(10)

,Fecha,cve_geo,Entidad,Municipio,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Incidencia,geometry,lon,lat
22864441,2025-09-01,20240,Oaxaca,San Martín Itunyoso,El patrimonio,Robo,Robo a institución bancaria,Con violencia,0.0,"POLYGON ((-97.83346 17.26365, -97.83145 17.263...",-97.855780,17.217359
13827764,2019-06-01,29011,Tlaxcala,Muñoz de Domingo Arenas,El patrimonio,Robo,Robo a institución bancaria,Sin violencia,0.0,"POLYGON ((-98.2406 19.53664, -98.23854 19.5336...",-98.233476,19.506080
3868711,2020-02-01,30101,Veracruz de Ignacio de la Llave,Mariano Escobedo,El patrimonio,Robo,Robo a negocio,Con violencia,0.0,"POLYGON ((-97.23029 18.97978, -97.2297 18.9791...",-97.183851,18.932973
19086476,2020-08-01,20114,Oaxaca,San Baltazar Yatzachi el Bajo,Otros bienes jurídicos afectados (del fuero co...,Contra el medio ambiente,Contra el medio ambiente,Contra el medio ambiente,0.0,"POLYGON ((-96.18186 17.25837, -96.18184 17.257...",-96.210610,17.220410
14639714,2023-06-01,12080,Guerrero,Juchitán,La familia,Violencia de género en todas sus modalidades d...,Violencia de género en todas sus modalidades d...,Violencia de género en todas sus modalidades d...,0.0,"POLYGON ((-98.70037 16.66215, -98.69906 16.661...",-98.677635,16.593979
19325765,2021-08-01,20086,Oaxaca,San Agustín Tlacotepec,El patrimonio,Robo,Robo de maquinaria,Robo de herramienta industrial o agrícola Con ...,0.0,"POLYGON ((-97.50914 17.23585, -97.50239 17.234...",-97.515741,17.198269
25198326,2024-10-01,20471,Oaxaca,Santiago Lalopa,El patrimonio,Abuso de confianza,Abuso de confianza,Abuso de confianza,0.0,"POLYGON ((-96.23783 17.44475, -96.2349 17.4461...",-96.250646,17.414526
29069537,2020-12-01,20221,Oaxaca,San Juan Teposcolula,Otros bienes jurídicos afectados (del fuero co...,Evasión de presos,Evasión de presos,Evasión de presos,0.0,"POLYGON ((-97.38327 17.65349, -97.38324 17.653...",-97.415758,17.587258
29071262,2020-12-01,20239,Oaxaca,San Martín Huamelúlpam,El patrimonio,Robo,Robo a transeúnte en vía pública,Sin violencia,0.0,"POLYGON ((-97.57022 17.4235, -97.56836 17.4232...",-97.608956,17.396296
12410601,2024-05-01,20279,Oaxaca,San Miguel Suchixtepec,El patrimonio,Extorsión,Extorsión,Extorsión,0.0,"POLYGON ((-96.44673 16.12729, -96.44584 16.126...",-96.458427,16.080168


In [21]:
end = gpd.GeoDataFrame(df_end)

In [ ]:
end.to_file('delitos.geojson', driver='GeoJSON')